In [1]:
from datasets import Dataset, load_dataset
from mlflow import MlflowClient
import mlflow

mlflow_tracking_uri="https://aipaas-mlflow.surromind.ai"
model_uri="models:/google-owlv2-base-patch16-ensemble/2"
model_name="google/owlv2-base-patch16-ensemble"
dataset_artifact_uri="mlflow-artifacts:/3/c6b9909566a349b1b0d11a97e796281d/artifacts/owlv2-dataset"
mlflow_experiment_name="ml_workflow_dev"

mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment(experiment_name=mlflow_experiment_name)
dataset_artifacts = mlflow.artifacts.download_artifacts(
    artifact_uri=dataset_artifact_uri
)

/Users/dhk/.local/share/virtualenvs/backend-Ujsw4nLg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_dataset = Dataset.from_csv(f"{dataset_artifacts}/train.csv")

Generating train split: 5297 examples [00:03, 1488.71 examples/s]


In [30]:
train_dataset[0]

{'image': '{\'bytes\': b\'\\xff\\xd8\\xff\\xe0\\x00\\x10JFIF\\x00\\x01\\x01\\x00\\x00\\x01\\x00\\x01\\x00\\x00\\xff\\xdb\\x00C\\x00\\x08\\x06\\x06\\x07\\x06\\x05\\x08\\x07\\x07\\x07\\t\\t\\x08\\n\\x0c\\x14\\r\\x0c\\x0b\\x0b\\x0c\\x19\\x12\\x13\\x0f\\x14\\x1d\\x1a\\x1f\\x1e\\x1d\\x1a\\x1c\\x1c $.\\\' ",#\\x1c\\x1c(7),01444\\x1f\\\'9=82<.342\\xff\\xdb\\x00C\\x01\\t\\t\\t\\x0c\\x0b\\x0c\\x18\\r\\r\\x182!\\x1c!22222222222222222222222222222222222222222222222222\\xff\\xc0\\x00\\x11\\x08\\x01w\\x01\\xf4\\x03\\x01"\\x00\\x02\\x11\\x01\\x03\\x11\\x01\\xff\\xc4\\x00\\x1f\\x00\\x00\\x01\\x05\\x01\\x01\\x01\\x01\\x01\\x01\\x00\\x00\\x00\\x00\\x00\\x00\\x00\\x00\\x01\\x02\\x03\\x04\\x05\\x06\\x07\\x08\\t\\n\\x0b\\xff\\xc4\\x00\\xb5\\x10\\x00\\x02\\x01\\x03\\x03\\x02\\x04\\x03\\x05\\x05\\x04\\x04\\x00\\x00\\x01}\\x01\\x02\\x03\\x00\\x04\\x11\\x05\\x12!1A\\x06\\x13Qa\\x07"q\\x142\\x81\\x91\\xa1\\x08#B\\xb1\\xc1\\x15R\\xd1\\xf0$3br\\x82\\t\\n\\x16\\x17\\x18\\x19\\x1a%&\\\'()*456789:CDEFGHIJSTUVWXYZcde

In [6]:
import ast
import io
from tqdm.auto import tqdm
import re
from PIL import Image
import numpy as np
from numpy import array  # array 함수를 글로벌 네임스페이스에 추가
def convert_array_string_to_dict(input_string):
    """
    NumPy array 문자열을 파이썬 딕셔너리로 변환
    
    Args:
        input_string (str): 변환할 문자열
    Returns:
        dict: 변환된 딕셔너리
    """
    
    # 문자열을 실행하여 딕셔너리 생성
    array_dict = eval(input_string)
    
    # 각 값을 파이썬 리스트로 변환
    result = {}
    for key, value in array_dict.items():
        if isinstance(value, np.ndarray):
            # object dtype인 경우 재귀적으로 처리
            if value.dtype == object:
                result[key] = [item.tolist() if isinstance(item, np.ndarray) else item for item in value]
            else:
                result[key] = value.tolist()
        else:
            result[key] = value
            
    return result
    
def convert_dataset_images(dataset):
    """
    일반 for문을 사용하여 데이터셋의 이미지를 변환합니다.
    """
    converted_data = []
    
    for item in tqdm(dataset, desc="Converting images"):
        try:
            # 현재 아이템의 모든 필드 복사
            converted_item = dict(item)
            
            # 이미지 변환
            img_dict = ast.literal_eval(item['image'])
            img_bytes = eval(img_dict['bytes']) if isinstance(img_dict['bytes'], str) else img_dict['bytes']
            converted_item['image'] = Image.open(io.BytesIO(img_bytes))
            converted_item['objects'] = convert_array_string_to_dict(item['objects'])
            
            converted_data.append(converted_item)
        except Exception as e:
            print(f"Error: {e}")
            converted_data.append(item)  # 에러 발생 시 원본 데이터 유지
    
    # 새로운 데이터셋 생성
    return Dataset.from_list(converted_data)

In [7]:
converted_dataset = convert_dataset_images(train_dataset)

Converting images:   0%|          | 0/5297 [00:00<?, ?it/s]

Converting images: 100%|██████████| 5297/5297 [00:02<00:00, 1795.91it/s]


In [34]:
converted_dataset[0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x375>,
 'image_id': 1,
 'width': 500,
 'height': 375,
 'objects': {'area': [3068.0, 690.0],
  'bbox': [[178.0, 84.0, 52.0, 59.0], [111.0, 144.0, 23.0, 30.0]],
  'category': ['helmet', 'helmet'],
  'id': [1, 1]}}